# 01: LangSmith Setup & Tracing

This notebook covers setting up LangSmith for AI pipeline observability, configuring LangChain integration, and understanding trace structure.

## Objectives
- Set up LangSmith account and API key
- Configure LangChain to use LangSmith
- Run a simple chain with tracing
- View and analyze traces in LangSmith dashboard

## 1. LangSmith Account Setup

### Prerequisites
1. Create a free account at [smith.langchain.com](https://smith.langchain.com)
2. Navigate to **Settings → API Keys**
3. Generate a new API key
4. Store the key securely (we'll use environment variables)

In [ ]:
# Install required packages
!pip install langsmith langchain langchain-openai python-dotenv

In [ ]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file (if present)
load_dotenv()

# For this notebook, we'll set them directly
# In production, use environment variables or a secrets manager

# TODO: Replace with your actual API keys
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = "your-langsmith-api-key"  # Replace with your key
os.environ["LANGCHAIN_PROJECT"] = "ai-engineering-course"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"

# OpenAI API key for the LLM calls
os.environ["OPENAI_API_KEY"] = "your-openai-api-key"  # Replace with your key

In [ ]:
# Verify LangSmith connection
from langsmith import Client

try:
    client = Client()
    # Try to list projects to verify connection
    projects = list(client.list_projects(limit=5))
    print(f"✓ LangSmith connection successful!")
    print(f"  Found {len(projects)} projects")
except Exception as e:
    print(f"✗ Connection failed: {e}")
    print("  Please check your API key and try again.")

## 2. Configuring LangChain with LangSmith

Once environment variables are set, LangChain automatically traces all chains and runs.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Create a simple chain
prompt = ChatPromptTemplate.from_template(
    "You are a helpful assistant. Explain {topic} in 2-3 sentences."
)

llm = ChatOpenAI(model="gpt-4", temperature=0.7)
parser = StrOutputParser()

# Compose the chain
chain = prompt | llm | parser

# Run the chain - this will be automatically traced!
result = chain.invoke({"topic": "recursion in programming"})
print(result)

### Check Your LangSmith Dashboard

After running the cell above:
1. Go to [smith.langchain.com](https://smith.langchain.com)
2. Navigate to your project ("ai-engineering-course")
3. You should see the trace of your chain execution

The trace will show:
- **PromptTemplate** → Input formatting
- **ChatOpenAI** → LLM call with token usage and latency
- **StrOutputParser** → Output extraction

## 3. Running a Chain with Detailed Tracing

Let's create a more complex chain and examine the trace structure.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

# Multi-turn conversation chain
conversation_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful coding assistant. Be concise and provide code examples."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

conversation_chain = conversation_prompt | llm | parser

# First turn
response1 = conversation_chain.invoke({
    "input": "What is a Python decorator?",
    "history": []
})
print("User: What is a Python decorator?")
print(f"Assistant: {response1}\n")

# Second turn (with history)
response2 = conversation_chain.invoke({
    "input": "Can you show me an example?",
    "history": [
        HumanMessage(content="What is a Python decorator?"),
        AIMessage(content=response1)
    ]
})
print("User: Can you show me an example?")
print(f"Assistant: {response2}")

## 4. Understanding Trace Structure

LangSmith traces contain detailed information about each step. Let's examine how to access trace data programmatically.

In [ ]:
from langsmith import Client

client = Client()

# Get recent runs from your project
runs = list(client.list_runs(
    project_name="ai-engineering-course",
    execution_order=1,  # Root runs only
    limit=5
))

print(f"Found {len(runs)} recent runs\n")

for run in runs:
    print(f"Run ID: {run.id}")
    print(f"  Name: {run.name}")
    print(f"  Status: {run.status}")
    print(f"  Duration: {run.total_tokens if run.total_tokens else 'N/A'} tokens")
    if run.total_tokens:
        print(f"  Total Tokens: {run.total_tokens}")
    print()

In [ ]:
# Examine child runs (nested steps)
if runs:
    latest_run = runs[0]
    child_runs = list(client.list_runs(
        parent_run_id=latest_run.id,
        project_name="ai-engineering-course"
    ))
    
    print(f"Run: {latest_run.name}")
    print(f"  Child runs: {len(child_runs)}\n")
    
    for child in child_runs:
        print(f"  └─ {child.name}")
        print(f"     Status: {child.status}")
        if child.input:
            input_preview = str(child.input)[:100] + "..." if len(str(child.input)) > 100 else str(child.input)
            print(f"     Input: {input_preview}")
        if child.output:
            output_preview = str(child.output)[:100] + "..." if len(str(child.output)) > 100 else str(child.output)
            print(f"     Output: {output_preview}")
        if child.total_tokens:
            print(f"     Tokens: {child.total_tokens}")
        print()

## 5. Using Custom Metadata and Tags

Add metadata to traces for better organization and filtering.

In [ ]:
from langsmith import traceable

@traceable(
    name="custom-research-chain",
    tags=["research", "v1"],
    metadata={"user_id": "student-001", "course": "ai-engineering"}
)
def research_chain(topic: str, depth: str = "brief") -> str:
    """A research chain with custom metadata."""
    
    @traceable(name="generate-questions")
    def generate_questions(topic: str) -> list[str]:
        prompt = f"Generate 3 research questions about {topic}"
        response = llm.invoke(prompt)
        return response.content.split("\n")
    
    @traceable(name="answer-questions")
    def answer_questions(questions: list[str]) -> str:
        answers = []
        for q in questions[:3]:
            if q.strip():
                response = llm.invoke(f"Brief answer: {q}")
                answers.append(f"Q: {q}\nA: {response.content}\n")
        return "\n".join(answers)
    
    # Execute the chain
    questions = generate_questions(topic)
    result = answer_questions(questions)
    
    return result

# Run with custom metadata
result = research_chain("large language models")
print(result[:500])  # Preview

## 6. Practical Exercise

### Task
1. Create a chain that takes a programming concept and generates:
   - A brief explanation
   - A code example
   - Common pitfalls

2. Use `@traceable` decorator with custom metadata

3. View the trace in LangSmith and identify:
   - Total token usage
   - Execution time
   - Input/output for each step

In [ ]:
# Your code here
# Create a multi-step chain and trace it

# @traceable(name="concept-explainer", tags=["educational"])
# def explain_concept(concept: str) -> dict:
#     ...
#     return {"explanation": ..., "code": ..., "pitfalls": ...}

## Summary

In this notebook, you learned:

1. **LangSmith Setup** — Creating an account and configuring API keys
2. **LangChain Integration** — Automatic tracing via environment variables
3. **Trace Structure** — Understanding nested runs and their components
4. **Custom Metadata** — Using `@traceable` decorator for better organization
5. **Programmatic Access** — Querying traces via the LangSmith client

### Next Steps
- Explore the LangSmith dashboard to visualize traces
- Try tracing more complex chains with tool usage
- Proceed to Notebook 02 for metrics monitoring